In [1]:
# ============================================================
# KANNADA RAG NOTEBOOK
# MODEL: google/muril-base-cased
# PIPELINE: Load -> Clean -> Chunk -> Retrieve -> Extractive QA -> Evaluate
# ============================================================

# =========================
# CELL 1 — INSTALL PACKAGES
# =========================
!pip -q install transformers accelerate faiss-cpu rank-bm25 pandas numpy tqdm torch

# =========================
# CELL 2 — IMPORTS
# =========================
import os
import re
import glob
import json
import unicodedata
from collections import Counter

import numpy as np
import pandas as pd
import torch
import faiss

from tqdm.auto import tqdm
from rank_bm25 import BM25Okapi
from transformers import AutoTokenizer, AutoModel, AutoModelForQuestionAnswering, pipeline

# =========================
# CELL 3 — DEVICE SETUP
# =========================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# =========================
# CELL 4 — MOUNT GOOGLE DRIVE (COLAB ONLY)
# =========================
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Not running in Colab or drive mount not needed.")

# =========================
# CELL 5 — PATHS
# =========================
CENTRAL_PATH = "/content/drive/MyDrive/AgriQADataset (1)/AgriQADataset/AgriQADataset/Central Govt Schemes"
STATE_PATH   = "/content/drive/MyDrive/AgriQADataset (1)/AgriQADataset/AgriQADataset/Karnataka State Schemes"

# If your folders are different, change these two paths only.

# =========================
# CELL 6 — TEXT CLEANING
# =========================
def clean_text(text: str) -> str:
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^\u0C80-\u0CFFa-zA-Z0-9.,!?()₹:/% \-]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

# =========================
# CELL 7 — LOAD DOCUMENTS
# =========================
central_files = sorted(glob.glob(os.path.join(CENTRAL_PATH, "*.txt")))
state_files   = sorted(glob.glob(os.path.join(STATE_PATH, "*.txt")))
all_files = central_files + state_files

print("Central files:", len(central_files))
print("State files:", len(state_files))
print("Total files:", len(all_files))

documents = []
for filepath in all_files:
    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        text = f.read()
    text = clean_text(text)
    if text:
        documents.append({
            "filename": os.path.basename(filepath),
            "path": filepath,
            "text": text
        })

print("Documents loaded:", len(documents))
print("Sample text:\n", documents[0]["text"][:1000] if documents else "No documents found")

# =========================
# CELL 8 — CHUNKING
# =========================
CHUNK_SIZE = 350
OVERLAP = 50

def chunk_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = OVERLAP):
    words = text.split()
    if len(words) <= chunk_size:
        return [" ".join(words)]

    chunks = []
    start = 0
    step = max(1, chunk_size - overlap)

    while start < len(words):
        chunk_words = words[start:start + chunk_size]
        if not chunk_words:
            break
        chunks.append(" ".join(chunk_words))
        start += step

    return chunks

chunked_docs = []
for doc in documents:
    chunks = chunk_text(doc["text"])
    for i, chunk in enumerate(chunks):
        chunked_docs.append({
            "filename": doc["filename"],
            "path": doc["path"],
            "chunk_id": i,
            "text": chunk
        })

print("Total chunks:", len(chunked_docs))

chunk_df = pd.DataFrame(chunked_docs)
chunk_df.head()

# =========================
# CELL 9 — LOAD MURIL FOR RETRIEVAL EMBEDDINGS
# =========================
RETRIEVAL_MODEL = "google/muril-base-cased"

retriever_tokenizer = AutoTokenizer.from_pretrained(RETRIEVAL_MODEL)
retriever_model = AutoModel.from_pretrained(RETRIEVAL_MODEL).to(DEVICE)
retriever_model.eval()

print("Retriever model loaded:", RETRIEVAL_MODEL)

# =========================
# CELL 10 — MEAN POOLING
# =========================
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    summed = torch.sum(token_embeddings * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts

# =========================
# CELL 11 — ENCODE TEXTS
# =========================
def encode_texts(texts, batch_size=8, max_length=256):
    all_embeddings = []

    for i in tqdm(range(0, len(texts), batch_size), desc="Encoding chunks"):
        batch_texts = texts[i:i + batch_size]
        inputs = retriever_tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        ).to(DEVICE)

        with torch.no_grad():
            outputs = retriever_model(**inputs)
            embeddings = mean_pooling(outputs, inputs["attention_mask"])
            embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)

        all_embeddings.append(embeddings.cpu().numpy())

    return np.vstack(all_embeddings)

texts = chunk_df["text"].tolist()
embeddings = encode_texts(texts, batch_size=8, max_length=256)
print("Embeddings shape:", embeddings.shape)

# =========================
# CELL 12 — FAISS INDEX
# =========================
dimension = embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(dimension)
faiss_index.add(embeddings)
print("FAISS index ready")

# =========================
# CELL 13 — BM25 INDEX
# =========================
tokenized_corpus = [text.split() for text in texts]
bm25 = BM25Okapi(tokenized_corpus)
print("BM25 ready")

# =========================
# CELL 14 — HYBRID RETRIEVAL
# =========================
def minmax_norm(arr):
    arr = np.array(arr, dtype=np.float32)
    if arr.size == 0:
        return arr
    mn = arr.min()
    mx = arr.max()
    return (arr - mn) / (mx - mn + 1e-8)

def hybrid_search(query, top_k=5, alpha=0.65, candidate_pool=30):
    """
    alpha -> dense weight
    (1 - alpha) -> BM25 weight
    """
    # Dense query embedding
    q_inputs = retriever_tokenizer(
        [query],
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    ).to(DEVICE)

    with torch.no_grad():
        q_outputs = retriever_model(**q_inputs)
        q_emb = mean_pooling(q_outputs, q_inputs["attention_mask"])
        q_emb = torch.nn.functional.normalize(q_emb, p=2, dim=1)

    q_emb_np = q_emb.cpu().numpy()

    # Dense scores
    dense_scores, dense_indices = faiss_index.search(q_emb_np, min(candidate_pool, len(texts)))
    dense_scores = dense_scores[0]
    dense_indices = dense_indices[0]

    # Sparse scores
    bm25_scores = np.array(bm25.get_scores(query.split()), dtype=np.float32)

    # Candidate set from dense + sparse
    dense_candidates = list(dense_indices[:candidate_pool])
    sparse_candidates = list(np.argsort(-bm25_scores)[:candidate_pool])
    candidate_ids = list(dict.fromkeys(dense_candidates + sparse_candidates))  # keep order, unique

    dense_dict = {idx: score for idx, score in zip(dense_indices, dense_scores)}

    dense_vals = np.array([dense_dict.get(idx, 0.0) for idx in candidate_ids], dtype=np.float32)
    sparse_vals = bm25_scores[candidate_ids]

    dense_vals_norm = minmax_norm(dense_vals)
    sparse_vals_norm = minmax_norm(sparse_vals)

    final_scores = alpha * dense_vals_norm + (1 - alpha) * sparse_vals_norm

    ranked_ids = [x for _, x in sorted(zip(final_scores, candidate_ids), reverse=True)]
    ranked_ids = ranked_ids[:top_k]

    results = []
    for idx in ranked_ids:
        results.append({
            "score": float(final_scores[candidate_ids.index(idx)]),
            "filename": chunk_df.iloc[idx]["filename"],
            "chunk_id": int(chunk_df.iloc[idx]["chunk_id"]),
            "text": texts[idx]
        })

    return results

# =========================
# CELL 15 — BUILD CONTEXT
# =========================
def build_context(results):
    parts = []
    for i, r in enumerate(results, start=1):
        parts.append(
            f"Document {i}\n"
            f"File: {r['filename']}\n"
            f"Chunk: {r['chunk_id']}\n"
            f"Text: {r['text']}"
        )
    return "\n\n".join(parts)

# =========================
# CELL 16 — LOAD EXTRACTIVE QA MODEL
# =========================
# NOTE:
# google/muril-base-cased is a base model. For QA quality, the best setup is to fine-tune
# this QA head on your own Kannada QA dataset.
# This notebook still loads it as an extractive QA pipeline so the full RAG flow works end-to-end.

QA_MODEL_NAME = "google/muril-base-cased"

qa_tokenizer = AutoTokenizer.from_pretrained(QA_MODEL_NAME)
qa_model = AutoModelForQuestionAnswering.from_pretrained(QA_MODEL_NAME).to(DEVICE)
qa_model.eval()

qa_pipe = pipeline(
    "question-answering",
    model=qa_model,
    tokenizer=qa_tokenizer,
    device=0 if torch.cuda.is_available() else -1
)

print("QA model loaded:", QA_MODEL_NAME)

# =========================
# CELL 17 — ANSWER GENERATION
# =========================
def answer_question(question, top_k_docs=5, score_threshold=0.10):
    """
    1) Retrieve top chunks
    2) Run extractive QA on each chunk
    3) Choose best answer by score
    """
    retrieved_docs = hybrid_search(question, top_k=top_k_docs)

    best = {
        "answer": "ಈ ಮಾಹಿತಿಯು ಲಭ್ಯವಿಲ್ಲ",
        "score": -1.0,
        "source": None
    }

    for doc in retrieved_docs:
        try:
            result = qa_pipe(question=question, context=doc["text"])
            answer = result.get("answer", "").strip()
            score = float(result.get("score", 0.0))

            if answer and score > best["score"]:
                best = {
                    "answer": answer,
                    "score": score,
                    "source": doc
                }
        except Exception as e:
            continue

    if best["score"] < score_threshold or not best["answer"]:
        best["answer"] = "ಈ ಮಾಹಿತಿಯು ಲಭ್ಯವಿಲ್ಲ"

    return best["answer"], retrieved_docs, best

# =========================
# CELL 18 — SINGLE TEST QUESTION
# =========================
question = "ಬೆಳೆ ಹಾನಿಗೆ ವಿಮೆ ನೀಡುವ ಯೋಜನೆ ಯಾವುದು?"
answer, retrieved_docs, best = answer_question(question)

print("=" * 100)
print("QUESTION:")
print(question)
print("\nANSWER:")
print(answer)
print("\nBEST SCORE:", best["score"])
print("\nSOURCE FILE:", best["source"]["filename"] if best["source"] else None)
print("=" * 100)

print("\nTOP RETRIEVED DOCUMENTS:\n")
for i, doc in enumerate(retrieved_docs, start=1):
    print(f"[{i}] {doc['filename']} | chunk {doc['chunk_id']} | score={doc['score']:.4f}")
    print(doc["text"][:500])
    print("-" * 100)

# =========================
# CELL 19 — INTERACTIVE LOOP
# =========================
while True:
    question = input("\nಪ್ರಶ್ನೆ ಕೇಳಿ (exit to stop): ").strip()
    if question.lower() == "exit":
        break

    answer, retrieved_docs, best = answer_question(question)

    print("\nಉತ್ತರ:", answer)
    print("Confidence:", round(best["score"], 4))
    if best["source"] is not None:
        print("Source:", best["source"]["filename"], "| chunk", best["source"]["chunk_id"])

# =========================
# CELL 20 — EVALUATION DATA
# =========================
# Replace this with your own evaluation set.
# Columns needed: question, ground_truth
evaluation_data = [
    {
        "question": "ಬೆಳೆ ವಿಮೆ ಯೋಜನೆ ಯಾವುದು?",
        "ground_truth": "ಪ್ರಧಾನಮಂತ್ರಿ ಫಸಲ್ ಬಿಮಾ ಯೋಜನೆ"
    },
    {
        "question": "ಅಪಘಾತ ವಿಮೆ ಯೋಜನೆ ಯಾವುದು?",
        "ground_truth": "ಕರ್ನಾಟಕ ರೈತ ಸುರಕ್ಷಾ ವಿಮೆ ಯೋಜನೆ"
    }
]

eval_df = pd.DataFrame(evaluation_data)
eval_df

# =========================
# CELL 21 — NORMALIZATION FOR METRICS
# =========================
def normalize_answer(s):
    s = unicodedata.normalize("NFKC", str(s))
    s = re.sub(r"[^\u0C80-\u0CFFa-zA-Z0-9 ]+", " ", s)
    s = s.lower()
    s = " ".join(s.split())
    return s

def exact_match(pred, gold):
    return int(normalize_answer(pred) == normalize_answer(gold))

def f1_score(pred, gold):
    pred_tokens = normalize_answer(pred).split()
    gold_tokens = normalize_answer(gold).split()

    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return 0.0

    common = Counter(pred_tokens) & Counter(gold_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)
    return (2 * precision * recall) / (precision + recall)

# =========================
# CELL 22 — RUN EVALUATION
# =========================
results = []

for _, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc="Evaluating"):
    q = row["question"]
    gold = row["ground_truth"]

    pred, retrieved_docs, best = answer_question(q)

    results.append({
        "question": q,
        "ground_truth": gold,
        "prediction": pred,
        "exact_match": exact_match(pred, gold),
        "f1": f1_score(pred, gold),
        "best_score": best["score"],
        "source_file": best["source"]["filename"] if best["source"] else None,
        "source_chunk": best["source"]["chunk_id"] if best["source"] else None
    })

results_df = pd.DataFrame(results)
results_df

# =========================
# CELL 23 — METRICS
# =========================
print("Exact Match:", results_df["exact_match"].mean())
print("F1:", results_df["f1"].mean())

# =========================
# CELL 24 — SAVE RESULTS
# =========================
results_df.to_csv("muril_rag_results.csv", index=False)
print("Saved: muril_rag_results.csv")

# =========================
# CELL 25 — SAVE CHUNK DATA
# =========================
chunk_df.to_csv("muril_chunks.csv", index=False)
print("Saved: muril_chunks.csv")

# =========================
# CELL 26 — OPTIONAL: SAVE RETRIEVAL INDEX META
# =========================
meta = {
    "retrieval_model": RETRIEVAL_MODEL,
    "qa_model": QA_MODEL_NAME,
    "chunk_size": CHUNK_SIZE,
    "overlap": OVERLAP,
    "num_documents": len(documents),
    "num_chunks": len(chunk_df)
}

with open("muril_rag_meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print("Saved: muril_rag_meta.json")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 51.3 MB/s eta 0:00:00
Device: cpu
Mounted at /content/drive
Central files: 40
State files: 42
Total files: 82
Documents loaded: 82
Sample text:
 ಯೋಜನೆ ಸಂಖ್ಯೆ : 1, ಯೋಜನೆ ಹೆಸರು : ಪ್ರಧಾನಮಂತ್ರಿ ಫಸಲ್ ಬಿಮಾ ಯೋಜನೆ , ಇಂಗ್ಲಿಷ್ ಹೆಸರು : Pradhan Mantri Fasal Bima Yojana (PMFBY) , ಉದ್ದೇಶ : ಪ್ರಧಾನಮಂತ್ರಿ ಫಸಲ್ ಬಿಮಾ ಯೋಜನೆವು ಬೆಳೆ ಹಾನಿಯಾದ ಸಂದರ್ಭಗಳಲ್ಲಿ ರೈತರಿಗೆ ವಿಮೆಯ ಮೂಲಕ ಆರ್ಥಿಕ ರಕ್ಷಣೆ ಒದಗಿಸಲು ರೂಪಿಸಲಾದ ಕೇಂದ್ರ ಸರ್ಕಾರದ ಮಹತ್ವದ ಕೃಷಿ ವಿಮಾ ಯೋಜನೆಯಾಗಿದೆ. ಪ್ರಕೃತಿ ವಿಪತ್ತುಗಳು, ಬರ, ಪ್ರವಾಹ, ಅತಿವೃಷ್ಟಿ, ಗಾಳಿ, ಮಳೆ ಕೊರತೆ, ಕೀಟ ಹಾಗೂ ರೋಗ ಹಾನಿ ಮುಂತಾದ ಕಾರಣಗಳಿಂದ ರೈತರು ಅನುಭವಿಸುವ ನಷ್ಟಕ್ಕೆ ಪರಿಹಾರ ನೀಡುವುದು ಇದರ ಪ್ರಮುಖ ಗುರಿಯಾಗಿದೆ. ಭಾರತದಲ್ಲಿ ಕೃಷಿ ಹವಾಮಾನಾಧಾರಿತವಾಗಿರುವುದರಿಂದ ರೈತರ ಆದಾಯ ಅಸ್ಥಿರವಾಗಿರುತ್ತದೆ. ಈ ಯೋಜನೆಯ ಮೂಲಕ ಕಡಿಮೆ ಪ್ರೀಮಿಯಂ ದರದಲ್ಲಿ ಹೆಚ್ಚಿನ ವಿಮಾ ರಕ್ಷಣೆ ಒದಗಿಸಲಾಗುತ್ತದೆ. ಬೆಳೆ ಕಟಾವು ಪರೀಕ್ಷೆಗಳು ಮತ್ತು ಹವಾಮಾನ ಮಾಹಿತಿಯ ಆಧಾರದ ಮೇಲೆ ನಷ್ಟವನ್ನು ಅಂದಾಜು ಮಾಡಿ ಪರಿಹಾರ ಮೊತ್ತವನ್ನು ನೇರವಾಗಿ ರೈತರ ಬ್ಯಾಂಕ್ ಖಾತೆಗೆ ಜಮಾ ಮಾಡಲಾಗುತ್ತದೆ. ಈ ಯೋಜನೆ ರೈತರನ್ನು ಆರ್ಥಿಕ ಅನಿಶ್ಚಿತತೆಯಿಂದ ರಕ್ಷಿಸಿ ಕೃಷಿಯಲ್ಲಿ ಮುಂದುವರಿಯಲು ಪ್ರ

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForQuestionAnswering LOAD REPORT from: google/muril-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
qa_outputs.bias                            | MISSING    | 
qa_outputs.weight                          | MIS

QA model loaded: google/muril-base-cased
QUESTION:
ಬೆಳೆ ಹಾನಿಗೆ ವಿಮೆ ನೀಡುವ ಯೋಜನೆ ಯಾವುದು?

ANSWER:
ಈ ಮಾಹಿತಿಯು ಲಭ್ಯವಿಲ್ಲ

BEST SCORE: 0.0001042145872816036

SOURCE FILE: 13.txt

TOP RETRIEVED DOCUMENTS:

[1] 1.txt | chunk 0 | score=0.8999
ಯೋಜನೆ ಸಂಖ್ಯೆ : 1, ಯೋಜನೆ ಹೆಸರು : ಪ್ರಧಾನಮಂತ್ರಿ ಫಸಲ್ ಬಿಮಾ ಯೋಜನೆ , ಇಂಗ್ಲಿಷ್ ಹೆಸರು : Pradhan Mantri Fasal Bima Yojana (PMFBY) , ಉದ್ದೇಶ : ಪ್ರಧಾನಮಂತ್ರಿ ಫಸಲ್ ಬಿಮಾ ಯೋಜನೆವು ಬೆಳೆ ಹಾನಿಯಾದ ಸಂದರ್ಭಗಳಲ್ಲಿ ರೈತರಿಗೆ ವಿಮೆಯ ಮೂಲಕ ಆರ್ಥಿಕ ರಕ್ಷಣೆ ಒದಗಿಸಲು ರೂಪಿಸಲಾದ ಕೇಂದ್ರ ಸರ್ಕಾರದ ಮಹತ್ವದ ಕೃಷಿ ವಿಮಾ ಯೋಜನೆಯಾಗಿದೆ. ಪ್ರಕೃತಿ ವಿಪತ್ತುಗಳು, ಬರ, ಪ್ರವಾಹ, ಅತಿವೃಷ್ಟಿ, ಗಾಳಿ, ಮಳೆ ಕೊರತೆ, ಕೀಟ ಹಾಗೂ ರೋಗ ಹಾನಿ ಮುಂತಾದ ಕಾರಣಗಳಿಂದ ರೈತರು ಅನುಭವಿಸುವ ನಷ್ಟಕ್ಕೆ ಪರಿಹಾರ ನೀಡುವುದು ಇದರ ಪ್ರಮುಖ ಗುರಿಯಾಗಿದೆ. ಭಾರತದಲ್ಲಿ ಕೃಷಿ ಹವಾಮಾನಾಧಾರಿತವಾಗಿರುವುದರಿಂದ ರೈತರ ಆದಾಯ ಅಸ್ಥಿರ
----------------------------------------------------------------------------------------------------
[2] 12.txt | chunk 0 | score=0.8869
ಯೋಜನೆ ಸಂಖ್ಯೆ: 12 ಯೋಜನೆ ಹೆಸರು: ಕಿಸಾನ್ ಕ್ರೆಡಿಟ್ ಕಾರ್ಡ್ ಯೋಜನೆ ಇಂಗ್ಲಿಷ್ ಹೆಸರು: Kisan Credit Card (KCC) Scheme ಉದ್ದೇಶ: ಕೃಷಿ ಮತ್ತು ಸ

Evaluating:   0%|          | 0/2 [00:00<?, ?it/s]

Exact Match: 0.0
F1: 0.0
Saved: muril_rag_results.csv
Saved: muril_chunks.csv
Saved: muril_rag_meta.json
